# 德译英机器翻译 - Transformer 演示

本 notebook 展示如何使用标准 Transformer 模型进行德语到英语的机器翻译。
模型基于 "Attention Is All You Need" 论文实现，使用 Multi30k 数据集训练。

**完全自包含，无需外部模块导入。**

## 1. 环境设置

In [ ]:
import os
import math
import urllib.request
import gzip
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

## 2. 配置参数

In [ ]:
# 模型配置 (基于论文的 base 配置)
config = {
    'data_dir': './data',
    
    # 模型参数
    'd_model': 512,
    'nhead': 8,
    'd_ff': 2048,
    'num_encoder_layers': 6,
    'num_decoder_layers': 6,
    'dropout': 0.1,
    'max_len': 128,
    
    # 训练参数
    'batch_size': 64,
    'epochs': 20,
    'lr': 1.0,
    'weight_decay': 1e-4,
    'warmup_steps': 4000,
    'grad_clip_norm': 1.0,
    'early_stopping_patience': 5,
}

print('Configuration:')
for k, v in config.items():
    print(f'  {k}: {v}')

## 3. 数据处理模块

In [ ]:
class Vocabulary:
    """词汇表"""

    def __init__(self):
        self.word2idx = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.idx2word = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
        self.pad_idx = 0
        self.sos_idx = 1
        self.eos_idx = 2
        self.unk_idx = 3

    def add_sentence(self, sentence: str) -> None:
        for word in sentence.strip().split():
            if word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word

    def add_word(self, word: str) -> None:
        if word not in self.word2idx:
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx] = word

    def encode(self, sentence: str) -> list:
        return [self.sos_idx] + [self.word2idx.get(w, self.unk_idx) for w in sentence.strip().split()] + [self.eos_idx]

    def decode(self, indices: list) -> str:
        words = [self.idx2word.get(i, "<unk>") for i in indices]
        words = [w for w in words if w not in ["<pad>", "<sos>", "<eos>"]]
        return " ".join(words)

    def __len__(self) -> int:
        return len(self.word2idx)

In [ ]:
class TranslationDataset(Dataset):
    """翻译数据集"""

    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=512):
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.src_sentences)

    def __getitem__(self, idx: int) -> dict:
        src = self.src_vocab.encode(self.src_sentences[idx])[:self.max_len]
        tgt = self.tgt_vocab.encode(self.tgt_sentences[idx])[:self.max_len]
        return {
            "src": torch.tensor(src, dtype=torch.long),
            "tgt": torch.tensor(tgt, dtype=torch.long),
        }


def collate_fn(batch, pad_idx=0):
    """批处理填充"""
    max_src_len = max(item["src"].size(0) for item in batch)
    max_tgt_len = max(item["tgt"].size(0) for item in batch)

    src_batch = torch.zeros(len(batch), max_src_len, dtype=torch.long).fill_(pad_idx)
    tgt_batch = torch.zeros(len(batch), max_tgt_len, dtype=torch.long).fill_(pad_idx)

    for i, item in enumerate(batch):
        src_batch[i, :item["src"].size(0)] = item["src"]
        tgt_batch[i, :item["tgt"].size(0)] = item["tgt"]

    return {"src": src_batch, "tgt": tgt_batch}

In [ ]:
def download_multi30k(data_dir):
    """下载 Multi30k 数据集 (德-英)"""
    os.makedirs(data_dir, exist_ok=True)

    base_url = "https://raw.githubusercontent.com/multi30k/dataset/master/data/task1/raw/"
    files = {
        "train.de": "train.de.gz",
        "train.en": "train.en.gz",
        "val.de": "val.de.gz",
        "val.en": "val.en.gz",
    }

    data = {}
    for name, remote in files.items():
        local_path = os.path.join(data_dir, name)
        gz_path = os.path.join(data_dir, remote)

        if not os.path.exists(local_path):
            print(f"Downloading {remote}...")
            urllib.request.urlretrieve(base_url + remote, gz_path)
            with gzip.open(gz_path, "rb") as f_in:
                with open(local_path, "wb") as f_out:
                    f_out.write(f_in.read())
            os.remove(gz_path)

        with open(local_path, "r", encoding="utf-8") as f:
            data[name] = [line.strip() for line in f.readlines()]

    return data["train.de"], data["train.en"], data["val.de"], data["val.en"]


def build_vocab(sentences, min_freq=2):
    """构建词汇表"""
    vocab = Vocabulary()
    word_counts = Counter()
    for sent in sentences:
        word_counts.update(sent.strip().split())

    for word, count in word_counts.items():
        if count >= min_freq:
            vocab.add_word(word)

    return vocab


def get_dataloaders(data_dir, batch_size, max_len=128, vocab_min_freq=2):
    """获取数据加载器"""
    train_de, train_en, val_de, val_en = download_multi30k(data_dir)

    print(f"Train samples: {len(train_de)}")
    print(f"Val samples: {len(val_de)}")

    src_vocab = build_vocab(train_de, vocab_min_freq)
    tgt_vocab = build_vocab(train_en, vocab_min_freq)

    print(f"Source vocab (de): {len(src_vocab)} words")
    print(f"Target vocab (en): {len(tgt_vocab)} words")

    train_dataset = TranslationDataset(train_de, train_en, src_vocab, tgt_vocab, max_len)
    val_dataset = TranslationDataset(val_de, val_en, src_vocab, tgt_vocab, max_len)

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        collate_fn=lambda b: collate_fn(b, src_vocab.pad_idx)
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=lambda b: collate_fn(b, src_vocab.pad_idx)
    )

    return train_loader, val_loader, src_vocab, tgt_vocab

In [ ]:
# 加载数据
train_loader, val_loader, src_vocab, tgt_vocab = get_dataloaders(
    data_dir=config['data_dir'],
    batch_size=config['batch_size'],
    max_len=config['max_len'],
)

print(f'\n训练集批次数: {len(train_loader)}')
print(f'验证集批次数: {len(val_loader)}')

In [ ]:
# 展示样本
def show_samples(loader, src_vocab, tgt_vocab, num_samples=5):
    batch = next(iter(loader))
    src = batch['src']
    tgt = batch['tgt']
    
    print('样本展示:')
    print('-' * 80)
    for i in range(min(num_samples, src.size(0))):
        src_text = src_vocab.decode(src[i].tolist())
        tgt_text = tgt_vocab.decode(tgt[i].tolist())
        print(f'德语: {src_text}')
        print(f'英语: {tgt_text}')
        print('-' * 80)

show_samples(train_loader, src_vocab, tgt_vocab)

## 4. Transformer 模型定义

In [ ]:
class PositionalEncoding(nn.Module):
    """正弦位置编码"""

    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [ ]:
class MultiHeadAttention(nn.Module):
    """多头注意力机制"""

    def __init__(self, d_model, nhead, dropout=0.1):
        super().__init__()
        assert d_model % nhead == 0
        self.d_k = d_model // nhead
        self.nhead = nhead

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        q = self.w_q(query).view(batch_size, -1, self.nhead, self.d_k).transpose(1, 2)
        k = self.w_k(key).view(batch_size, -1, self.nhead, self.d_k).transpose(1, 2)
        v = self.w_v(value).view(batch_size, -1, self.nhead, self.d_k).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.nhead * self.d_k)
        return self.w_o(out)

In [ ]:
class PositionwiseFeedForward(nn.Module):
    """前馈网络"""

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w_2(self.dropout(F.relu(self.w_1(x))))

In [ ]:
class EncoderLayer(nn.Module):
    """编码器层"""

    def __init__(self, d_model, nhead, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        attn_out = self.self_attn(x, x, x, src_mask)
        x = self.norm1(x + self.dropout1(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))
        return x

In [ ]:
class DecoderLayer(nn.Module):
    """解码器层"""

    def __init__(self, d_model, nhead, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.cross_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, memory, tgt_mask=None, memory_mask=None):
        attn_out = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(attn_out))
        attn_out = self.cross_attn(x, memory, memory, memory_mask)
        x = self.norm2(x + self.dropout2(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout3(ffn_out))
        return x

In [ ]:
class Encoder(nn.Module):
    """编码器"""

    def __init__(self, vocab_size, d_model, nhead, d_ff, num_layers, dropout=0.1, max_len=5000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, nhead, d_ff, dropout) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, src, src_mask=None):
        x = self.embedding(src) * math.sqrt(self.embedding.embedding_dim)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.norm(x)

In [ ]:
class Decoder(nn.Module):
    """解码器"""

    def __init__(self, vocab_size, d_model, nhead, d_ff, num_layers, dropout=0.1, max_len=5000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, nhead, d_ff, dropout) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        x = self.embedding(tgt) * math.sqrt(self.embedding.embedding_dim)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, memory, tgt_mask, memory_mask)
        return self.norm(x)

In [ ]:
class Transformer(nn.Module):
    """完整 Transformer 模型"""

    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        nhead=8,
        d_ff=2048,
        num_encoder_layers=6,
        num_decoder_layers=6,
        dropout=0.1,
        max_len=5000,
    ):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, nhead, d_ff, num_encoder_layers, dropout, max_len)
        self.decoder = Decoder(tgt_vocab_size, d_model, nhead, d_ff, num_decoder_layers, dropout, max_len)
        self.generator = nn.Linear(d_model, tgt_vocab_size)

        self._init_parameters()

    def _init_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, memory_mask=None):
        memory = self.encoder(src, src_mask)
        output = self.decoder(tgt, memory, tgt_mask, memory_mask)
        return self.generator(output)

    def encode(self, src, src_mask=None):
        return self.encoder(src, src_mask)

    def decode(self, tgt, memory, tgt_mask=None, memory_mask=None):
        return self.decoder(tgt, memory, tgt_mask, memory_mask)

In [ ]:
# 创建模型
model = Transformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    d_model=config['d_model'],
    nhead=config['nhead'],
    d_ff=config['d_ff'],
    num_encoder_layers=config['num_encoder_layers'],
    num_decoder_layers=config['num_decoder_layers'],
    dropout=config['dropout'],
    max_len=config['max_len'],
).to(device)

print(model)
print(f'\n模型参数量: {sum(p.numel() for p in model.parameters()):,}')

## 5. 训练工具

In [ ]:
class WarmupLRScheduler(optim.lr_scheduler._LRScheduler):
    """带 warmup 的学习率调度器
    
    lr = d_model^(-0.5) * min(step^(-0.5), step * warmup_steps^(-1.5))
    """

    def __init__(self, optimizer, d_model, warmup_steps, last_epoch=-1):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        step = self._step_count
        scale = self.d_model ** (-0.5) * min(step ** (-0.5), step * self.warmup_steps ** (-1.5))
        return [base_lr * scale for base_lr in self.base_lrs]


class LabelSmoothingLoss(nn.Module):
    """标签平滑损失"""

    def __init__(self, vocab_size, padding_idx, smoothing=0.1):
        super().__init__()
        self.criterion = nn.CrossEntropyLoss(ignore_index=padding_idx, label_smoothing=smoothing)

    def forward(self, pred, target):
        pred = pred.contiguous().view(-1, pred.size(-1))
        target = target.contiguous().view(-1)
        return self.criterion(pred, target)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scheduler, device, pad_idx, grad_clip_norm=1.0):
    """训练一个 epoch"""
    model.train()
    total_loss = 0.0

    for batch in loader:
        src = batch["src"].to(device)
        tgt = batch["tgt"].to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        src_mask = (src != pad_idx).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt_input != pad_idx).unsqueeze(1).unsqueeze(2) & \
                   torch.tril(torch.ones(tgt_input.size(1), tgt_input.size(1), device=device)).bool()

        optimizer.zero_grad()
        output = model(src, tgt_input, src_mask, tgt_mask)
        loss = criterion(output, tgt_output)
        loss.backward()

        if grad_clip_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device, pad_idx):
    """评估模型"""
    model.eval()
    total_loss = 0.0

    for batch in loader:
        src = batch["src"].to(device)
        tgt = batch["tgt"].to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        src_mask = (src != pad_idx).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt_input != pad_idx).unsqueeze(1).unsqueeze(2) & \
                   torch.tril(torch.ones(tgt_input.size(1), tgt_input.size(1), device=device)).bool()

        output = model(src, tgt_input, src_mask, tgt_mask)
        loss = criterion(output, tgt_output)

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def greedy_decode(model, src, max_len, start_symbol, end_symbol, pad_idx, device):
    """贪婪解码"""
    model.eval()

    src_mask = (src != pad_idx).unsqueeze(1).unsqueeze(2)
    memory = model.encode(src, src_mask)

    ys = torch.ones(1, 1).fill_(start_symbol).long().to(device)

    for _ in range(max_len - 1):
        tgt_mask = torch.tril(torch.ones(ys.size(1), ys.size(1), device=device)).bool()
        tgt_mask = tgt_mask.unsqueeze(0).unsqueeze(0)

        out = model.decode(ys, memory, tgt_mask, src_mask)
        prob = model.generator(out[:, -1, :])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.item()

        ys = torch.cat([ys, torch.ones(1, 1).fill_(next_word).long().to(device)], dim=1)

        if next_word == end_symbol:
            break

    return ys

## 6. 训练

In [ ]:
# 损失函数和优化器
criterion = LabelSmoothingLoss(
    vocab_size=len(tgt_vocab),
    padding_idx=tgt_vocab.pad_idx,
    smoothing=0.1
)

optimizer = optim.Adam(
    model.parameters(),
    lr=config['lr'],
    betas=(0.9, 0.98),
    eps=1e-9,
    weight_decay=config['weight_decay']
)

scheduler = WarmupLRScheduler(
    optimizer,
    d_model=config['d_model'],
    warmup_steps=config['warmup_steps']
)

# 记录训练历史
history = {
    'train_loss': [],
    'val_loss': [],
    'lr': []
}

# Early stopping
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

In [ ]:
# 训练循环
print('开始训练...')

for epoch in range(config['epochs']):
    train_loss = train_epoch(
        model, train_loader, criterion, optimizer, scheduler,
        device, src_vocab.pad_idx, config['grad_clip_norm']
    )
    
    val_loss = evaluate(
        model, val_loader, criterion, device, src_vocab.pad_idx
    )
    
    current_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(current_lr)
    
    print(f'Epoch {epoch+1}/{config["epochs"]}: '
          f'Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | '
          f'LR: {current_lr:.6f}')
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= config['early_stopping_patience']:
            print(f'Early stopping at epoch {epoch+1}')
            break

# 加载最佳模型
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f'\n加载最佳模型 (Val Loss: {best_val_loss:.4f})')

## 7. 训练曲线可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss 曲线
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# 学习率曲线
axes[1].plot(history['lr'], label='Learning Rate', marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('Learning Rate Schedule')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 8. 交互式演示 - 德译英翻译

In [ ]:
def translate(sentence):
    """翻译德语句子到英语"""
    model.eval()
    
    src_indices = src_vocab.encode(sentence)
    src_tensor = torch.tensor([src_indices], dtype=torch.long, device=device)
    
    with torch.no_grad():
        output_indices = greedy_decode(
            model,
            src_tensor,
            max_len=config['max_len'],
            start_symbol=tgt_vocab.sos_idx,
            end_symbol=tgt_vocab.eos_idx,
            pad_idx=src_vocab.pad_idx,
            device=device
        )
    
    translation = tgt_vocab.decode(output_indices[0].tolist())
    return translation

# 测试翻译
test_sentences = [
    "ein mann steht auf einer straße",
    "eine frau liest ein buch",
    "zwei kinder spielen im park",
]

print('翻译测试:')
print('-' * 60)
for sent in test_sentences:
    translation = translate(sent)
    print(f'德语: {sent}')
    print(f'英语: {translation}')
    print('-' * 60)

In [ ]:
# 交互式翻译界面
input_text = widgets.Text(
    value='',
    placeholder='输入德语句子...',
    description='德语:',
    layout=widgets.Layout(width='500px')
)

translate_btn = widgets.Button(
    description='翻译',
    button_style='success',
    layout=widgets.Layout(width='100px')
)

output_area = widgets.Output()

# 示例按钮
example_btns = []
example_sentences = [
    "ein mann steht auf einer straße",
    "eine frau liest ein buch",
    "zwei kinder spielen im park",
    "der hund läuft im garten",
]

def on_example_click(btn):
    input_text.value = btn.description

for sent in example_sentences:
    btn = widgets.Button(
        description=sent,
        layout=widgets.Layout(width='auto')
    )
    btn.on_click(on_example_click)
    example_btns.append(btn)

def on_translate_click(btn):
    with output_area:
        clear_output()
        if input_text.value.strip():
            translation = translate(input_text.value)
            print(f'\n德语: {input_text.value}')
            print(f'英语: {translation}')
        else:
            print('请输入德语句子')

translate_btn.on_click(on_translate_click)

# 显示界面
display(widgets.HTML('<b>德译英翻译演示</b>'))
display(widgets.HTML('<p>点击示例或输入德语句子:</p>'))
display(widgets.HBox(example_btns))
display(widgets.HBox([input_text, translate_btn]))
display(output_area)

## 说明

1. **模型架构**: 标准 Transformer 编码器-解码器架构，遵循 "Attention Is All You Need" 论文
2. **训练**: 使用 Adam 优化器 + Warmup 学习率调度 + 标签平滑损失
3. **推理**: 使用贪婪解码生成翻译
4. **数据**: Multi30k 德语-英语数据集
5. **完全自包含**: 所有代码内嵌，无需外部模块导入